In [1]:
import os
import pandas as pd

# Use GPU 2
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["WANDB_MODE"] = "disabled"

import torch


from datasets import Dataset
from sklearn.metrics import classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments


# Settings
MODEL_NAME = 'roberta-base'
BASE_DIR = "/shared/4/projects/research-jam-2024/"
DATA_FP = '../../annotation/email_understanding/annotation_output/full/df4model.csv'

2024-06-11 20:21:00.174329: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
os.chdir('/shared/4/projects/research-jam-2024/')
train_df = pd.read_csv('data/final_training_data.tsv', sep='\t')
dev_df = pd.read_csv('data/final_validation_data.tsv', sep='\t')
test_df = pd.read_csv('data/final_test_data.tsv', sep='\t')

train_df['split'] = 'train'
dev_df['split'] = 'dev'
test_df['split'] = 'test'

df = pd.concat([train_df, dev_df, test_df])
print(len(train_df), len(dev_df), len(test_df))
df = df.rename({
    'message_id': 'id',
    'message_body_clean': 'text'
}, axis=1)

df.columns.tolist()

1448 181 182


['id',
 'text',
 'subject',
 'user',
 'Sharing',
 'Requesting',
 'Promising',
 'Personal',
 'Pleasantries',
 'Spam',
 'ResponseExpected',
 'split']

**Read and prepare the data**

In [15]:
def preprocess_data(examples):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenized_inputs = tokenizer(examples['text'], padding="max_length", max_length=512, truncation=True)
    
    label_columns = ['Sharing', 'Requesting', 'Promising', 'Personal', 'Pleasantries', 'Spam', 'ResponseExpected']
    labels = [[float(examples[label][i]) for i in range(len(examples['text']))] for label in label_columns]
    labels = torch.tensor(list(zip(*labels)), dtype=torch.float)
    
    tokenized_inputs["labels"] = labels
    
    return tokenized_inputs


train = df[df['split'] == 'train'].drop(['split', 'user', 'subject'], axis=1)
dev = df[df['split'] == 'dev'].drop(['split', 'user', 'subject'], axis=1)
test = df[df['split'] == 'test'].drop(['split', 'user', 'subject'], axis=1)

train_dataset = Dataset.from_pandas(train)
dev_dataset = Dataset.from_pandas(dev)
test_dataset = Dataset.from_pandas(test)

train_dataset = train_dataset.map(preprocess_data, batched=True)
dev_dataset = dev_dataset.map(preprocess_data, batched=True)
test_dataset = test_dataset.map(preprocess_data, batched=True)

# Make sure there are no empty labels
train_dataset = train_dataset.filter(lambda example: None not in example['labels'])
dev_dataset = dev_dataset.filter(lambda example: None not in example['labels'])
test_dataset = test_dataset.filter(lambda example: None not in example['labels'])

print(f"Train length: {len(train_dataset)}. Eval length: {len(dev_dataset)}. Test length: {len(test_dataset)}")
train.head(3)

Map:   0%|          | 0/1448 [00:00<?, ? examples/s]

Map:   0%|          | 0/181 [00:00<?, ? examples/s]

Map:   0%|          | 0/182 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1448 [00:00<?, ? examples/s]

Filter:   0%|          | 0/181 [00:00<?, ? examples/s]

Filter:   0%|          | 0/182 [00:00<?, ? examples/s]

Train length: 1448. Eval length: 181. Test length: 182


,id,text,Sharing,Requesting,Promising,Personal,Pleasantries,Spam,ResponseExpected
0,<cq3PHOfhJOTYC76k@example.com>,<gmane_tag_salutation> I have two question reg...,1,1,0,0,0,0,1
1,<VZjyVE+ZcYO+y1AP@example.com>,<gmane_tag_quotation_marker> <gmane_tag_quotat...,1,1,0,0,0,0,1
2,<y1o8Q+WXGhPnqsji@example.com>,Even better - they're listed as #2 in Forbes' ...,1,0,0,0,0,0,0


In [16]:
for example in train_dataset.take(1):
    print(example['labels'])

[1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0]


**Setup the pretrained model and datasets**

In [17]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, 
                                                           num_labels=7, 
                                                           problem_type="multi_label_classification")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [22]:
training_args = TrainingArguments(
    output_dir=f'/shared/4/projects/research-jam-2024/models/fine-tuned/{MODEL_NAME}/',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    evaluation_strategy="epoch",  # Evaluate at the end of each epoch
    logging_dir='./logs',
    save_strategy="epoch",  # Save a model checkpoint at the end of each epoch
    save_total_limit=1,  # Only keep the last checkpoint
    load_best_model_at_end=True,  # Load the best model at the end of training
    metric_for_best_model="eval_loss",  # Use eval_loss to identify the best model
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
)

trainer.train()

/opt/anaconda/lib/python3.10/site-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,No log,0.239241
2,No log,0.245569
3,0.155300,0.260712
4,0.155300,0.268978
5,0.155300,0.261760


TrainOutput(global_step=905, training_loss=0.13483431062645676, metrics={'train_runtime': 206.907, 'train_samples_per_second': 34.992, 'train_steps_per_second': 4.374, 'total_flos': 1905009558528000.0, 'train_loss': 0.13483431062645676, 'epoch': 5.0})

**Evaluate on the test set**

In [19]:
predictions, labels, _ = trainer.predict(test_dataset)
predictions = torch.sigmoid(torch.tensor(predictions)).numpy() > 0.5

labels = test_dataset["labels"]
predictions = predictions.astype(int)

# Generating classification report for each label
label_names = ['Sharing', 'Requesting', 'Promising', 'Personal', 'Pleasantries', 'Spam', 'ResponseExpected']
print("RoBERTa Base results")
print(classification_report(labels, predictions, target_names=label_names))

RoBERTa Base results
                  precision    recall  f1-score   support

         Sharing       0.88      0.91      0.89       150
      Requesting       0.81      0.89      0.85        76
       Promising       0.00      0.00      0.00         8
        Personal       0.00      0.00      0.00         5
    Pleasantries       0.43      0.50      0.46         6
            Spam       0.00      0.00      0.00         8
ResponseExpected       0.78      0.88      0.82        75

       micro avg       0.82      0.83      0.83       328
       macro avg       0.41      0.45      0.43       328
    weighted avg       0.77      0.83      0.80       328
     samples avg       0.80      0.80      0.78       328



/opt/anaconda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


**Save the results to the server**

In [23]:
output_file = f'/shared/4/projects/research-jam-2024/models/fine-tuned/{MODEL_NAME}/predictions.csv'

In [24]:
predictions_binary = (torch.sigmoid(torch.tensor(predictions)).numpy() > 0.5).astype(int)

# Create a DataFrame with the instance_id from the test set
test_instance_ids = test['id'].reset_index(drop=True)
predictions_df = pd.DataFrame(predictions_binary, columns=label_names)

# Combine instance IDs with predictions
final_predictions_df = pd.concat([test_instance_ids, predictions_df], axis=1)
final_predictions_df.to_csv(output_file, index=False)

print(f"Predictions saved to {output_file}")

Predictions saved to /shared/4/projects/research-jam-2024/models/fine-tuned/roberta-base/predictions.csv
